In [25]:
import os
import sys

if "google.colab" in sys.modules:
    # Running in Colab

    !git clone https://github.com/pthengtr/kcw-analytics.git
    !cd /content/kcw-analytics && git pull origin main

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"
else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

print("Using folder:", BASE_FOLDER)

fatal: destination path 'kcw-analytics' already exists and is not an empty directory.
From https://github.com/pthengtr/kcw-analytics
 * branch            main       -> FETCH_HEAD
Already up to date.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using folder: /content/drive/Shareddrives


In [26]:

folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw"

In [27]:
import os
import pandas as pd

data = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)
        data[file] = pd.read_csv(
            path,
            dtype={
              "BCODE": "string",
              "ITEMNO": "string",
              "BILLNO": "string",
            },
            encoding="utf-8-sig",
            low_memory=False   # stops chunk guessing
        )
        print(f"Loaded: {file} -> {data[file].shape}")



Loaded: raw_inventory_hq_2024.csv -> (4983, 8)
Loaded: raw_syp_pimas_purchase_bills.csv -> (2970, 49)
Loaded: raw_syp_pidet_purchase_lines.csv -> (27770, 41)
Loaded: raw_syp_simas_sales_bills.csv -> (12876, 49)
Loaded: raw_syp_sidet_sales_lines.csv -> (37824, 38)
Loaded: raw_hq_pimas_purchase_bills.csv -> (50274, 49)
Loaded: raw_hq_pidet_purchase_lines.csv -> (154052, 41)
Loaded: raw_hq_sidet_sales_lines.csv -> (733417, 38)
Loaded: raw_hq_icmas_products.csv -> (114950, 94)
Loaded: raw_hq_armas_receivable.csv -> (2227, 20)
Loaded: raw_hq_apmas_payable.csv -> (978, 20)
Loaded: raw_hq_simas_sales_bills.csv -> (275967, 49)
Loaded: raw_hq_pvmas_notes_vouchers.csv -> (13875, 32)


In [28]:

import sys
import importlib

# ensure repo is on path
repo_path = f"{BASE_FOLDER_GIT}/kcw-analytics"
if repo_path not in sys.path:
    sys.path.append(repo_path)

# import the module (NOT individual functions)
import src.kcw.utils as utils

# reload to pick up latest .py changes
importlib.reload(utils)



<module 'src.kcw.utils' from '/content/kcw-analytics/src/kcw/utils.py'>

In [29]:
df_simas = data["raw_hq_simas_sales_bills.csv"].copy()
df_pimas = data["raw_hq_pimas_purchase_bills.csv"].copy()

In [30]:
df_simas.columns

Index(['ID', 'JOURMODE', 'JOURTYPE', 'JOURDATE', 'JOURNO', 'JOURTIME',
       'DEPTNO', 'BOOKNO', 'BILLTYPE', 'BILLDATE', 'BILLTIME', 'BILLNO',
       'LINES', 'TAXIC', 'DISCOUNT', 'DEDUCT', 'BEFORETAX', 'VAT', 'TAX',
       'AFTERTAX', 'EXEMPT', 'SVCCHG', 'WITHHOLD', 'PAID', 'CASHED', 'CASHAMT',
       'CHKAMT', 'DUEAMT', 'PAYSTAT', 'ACCTNO', 'ACCTNAME', 'ADDR1', 'ADDR2',
       'PO', 'SALE', 'RE', 'TERM', 'DUEDATE', 'NOTEDATE', 'NOTENO',
       'VOUCDATE1', 'VOUCNO1', 'VOUCDATE2', 'VOUCNO2', 'POSTED1', 'POSTED2',
       'REMARKS', 'CANCELED', 'DONE'],
      dtype='object')

In [31]:
cutoff = pd.Timestamp.today().replace(day=1)

YEAR = cutoff.year
MONTH = cutoff.month

cutoff

Timestamp('2026-03-01 22:42:00.468838')

In [32]:
def normalize_acctname(name):
    if pd.isna(name):
        return name

    n = str(name).lower()

    for k in ["lazada", "shopee", "tiktok"]:
        if k in n:
            return f"คุณลูกค้าทั่วไป {k}"

    return name


In [33]:
df = df_simas

df["BILLDATE"] = pd.to_datetime(df["BILLDATE"].astype(str).str.strip(), errors="coerce")
df["VOUCDATE2"] = pd.to_datetime(df["VOUCDATE2"], errors="coerce")

mask_billno = df["BILLNO"].str.contains(r"TD|TR|TAD|CN", na=False)
mask_due_pos = df["DUEAMT"] != 0
mask_billbefore = df["BILLDATE"] < cutoff
mask_paidafter = df["VOUCDATE2"] > cutoff
mask_null = df["VOUCDATE2"].isna()
mask_cancel = df["CANCELED"].astype(str).str.strip().str.upper() == "N"
mask_vat = pd.to_numeric(df["VAT"], errors="coerce").fillna(0) > 0

df["ACCTNAME"] = df["ACCTNAME"].apply(normalize_acctname)

df_ar_sale = df[mask_billno & mask_due_pos & mask_billbefore & (mask_null | mask_paidafter) & mask_cancel & mask_vat]

df_ar_sale = df_ar_sale.sort_values(by=["ACCTNAME", "BILLDATE"])

In [34]:
df_ar_sale[["BILLNO", "BILLDATE","ACCTNO", "ACCTNAME", "VOUCDATE1", "VOUCDATE2", "PAID","DUEAMT", "VAT"]]

,BILLNO,BILLDATE,ACCTNO,ACCTNAME,VOUCDATE1,VOUCDATE2,PAID,DUEAMT,VAT
272995,TD6902-095,2026-02-16,7ออช,คุญ อัญชลี สายสอน,NaN,NaT,N,6190.0,7.0
272996,CN6902-017,2026-02-16,7ออช,คุญ อัญชลี สายสอน,NaN,NaT,N,-6190.0,7.0
273001,TD6902-098,2026-02-16,7ชลล,คุณ ชาลี สุพรรณ,NaN,NaT,N,400.0,7.0
273002,CN6902-020,2026-02-16,7ชลล,คุณ ชาลี สุพรรณ,NaN,NaT,N,-400.0,7.0
270884,TD6901-132,2026-01-31,7ธรพ,คุณ ธีรพงศ์ ฤทธิไกร,NaN,NaT,N,950.0,7.0
...,...,...,...,...,...,...,...,...,...
274258,TD6902-138,2026-02-24,7กช,หจก. ส.กิจชัยคอนกรีต (สาขาที่ 00002),NaN,NaT,N,3680.0,7.0
249175,TD6810-003,2025-10-01,7จยน,ห้างหุ้นส่วนจำกัด จี.ยูเนี่ยน (สำนักงานใหญ่),NaN,2026-03-04,Y,18860.0,7.0
261324,TD6812-052,2025-12-10,7จยน,ห้างหุ้นส่วนจำกัด จี.ยูเนี่ยน (สำนักงานใหญ่),NaN,2026-03-04,Y,25720.0,7.0
268518,TD6901-080,2026-01-21,7จยน,ห้างหุ้นส่วนจำกัด จี.ยูเนี่ยน (สำนักงานใหญ่),NaN,2026-03-04,Y,3200.0,7.0


In [35]:
df = df_ar_sale

df_ar_summary = (
    df
    .groupby("ACCTNAME")
    .agg(
        bills=("BILLNO", "nunique"),
        total_amount=("DUEAMT", "sum")
    )
    .reset_index()
)

In [36]:
df_ar_summary

,ACCTNAME,bills,total_amount
0,คุญ อัญชลี สายสอน,2,0.00
1,คุณ ชาลี สุพรรณ,2,0.00
2,คุณ ธีรพงศ์ ฤทธิไกร,2,0.00
3,คุณ นุชรีย์ รอดยัง,2,0.00
4,คุณ บุญธรรม สามารถ,2,0.00
5,คุณ ปรเมษฐ์ แสงสุรี,2,0.00
6,คุณ ยุทธสิทธิ์ วัณกลาง,2,0.00
7,คุณ วงษ์สวัสดิ์ จำปาทอง,2,0.00
8,คุณ ศรราม น้ำเพชร,2,0.00
9,คุณ ศิริวุฒิ โสตถิรัตนพันธ์,2,0.00


In [37]:
df = df_pimas

df["BILLDATE"] = pd.to_datetime(df["BILLDATE"].astype(str).str.strip(), errors="coerce")
df["VOUCDATE2"] = pd.to_datetime(df["VOUCDATE2"], errors="coerce")

mask_due_pos = df["DUEAMT"] != 0
mask_billbefore = df["BILLDATE"] < cutoff
mask_paidafter = df["VOUCDATE2"] > cutoff
mask_null = df["VOUCDATE2"].isna()
mask_cancel = df["CANCELED"].astype(str).str.strip().str.upper() == "N"
mask_vat = pd.to_numeric(df["VAT"], errors="coerce").fillna(0) > 0

df["ACCTNAME"] = df["ACCTNAME"].apply(normalize_acctname)

df_ap_purchase = df[mask_due_pos & mask_billbefore & (mask_null | mask_paidafter) & mask_cancel & mask_vat]

df_ap_purchase = df_ap_purchase.sort_values(by=["ACCTNAME", "BILLDATE"])

In [38]:
df_ap_purchase[["BILLNO", "BILLDATE","ACCTNO", "ACCTNAME", "VOUCDATE1", "VOUCDATE2", "PAID","DUEAMT", "VAT"]]

,BILLNO,BILLDATE,ACCTNO,ACCTNAME,VOUCDATE1,VOUCDATE2,PAID,DUEAMT,VAT
49197,IV69012611,2026-01-26,7PRW,บจก. ประวิทย์ ออโตพาร์ท อิมปอร์ต-เอ็กซปอร์ต (ส...,NaN,NaT,N,4010.36,7.0
48466,BV6901000002,2026-01-06,7MSC,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,NaN,NaT,N,8988.75,7.0
48526,BV6901000003,2026-01-07,7MSC,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,NaN,NaT,N,1239.97,7.0
48577,BV6901000005,2026-01-08,7MSC,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,NaN,NaT,N,2994.88,7.0
48890,BV6901000012,2026-01-17,7MSC,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,NaN,NaT,N,5569.56,7.0
...,...,...,...,...,...,...,...,...,...
49549,IV26020220,2026-02-07,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,4721.91,7.0
49569,IV26020239,2026-02-09,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,3358.97,7.0
49710,IV26020316,2026-02-12,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,3608.76,7.0
49872,IV26020412,2026-02-20,7LK,ห้างหุ้นส่วนจำกัด โลหะกิจกลการ อิมปอร์ต (สำนัก...,NaN,NaT,N,4660.12,7.0


In [39]:
df = df_ap_purchase

df_ap_summary = (
    df
    .groupby("ACCTNAME")
    .agg(
        bills=("BILLNO", "nunique"),
        total_amount=("DUEAMT", "sum")
    )
    .reset_index()
)

In [40]:
df_ap_summary

,ACCTNAME,bills,total_amount
0,บจก. ประวิทย์ ออโตพาร์ท อิมปอร์ต-เอ็กซปอร์ต (ส...,1,4010.36
1,บจก. มิตซู สแปร์พาร์ทส เอ็กซ์พอร์ทติ้ง (2012) ...,9,37372.11
2,บจก. เอส ที เค กรุ๊ป 2008 (สำนักงานใหญ่),21,48643.45
3,บจก. เอส.พี.อาร์.วาย ออโต้พาร์ท (สำนักงานใหญ่),22,23985.12
4,บจก.ศรีสยามกลการ (สาขาที่ 00000),74,433128.44
...,...,...,...
86,ห้างหุ้นส่วนจำกัด คิว.ซี.ออโตพาร์ท (สำนักงานใหญ่),21,48012.92
87,ห้างหุ้นส่วนจำกัด พี.อี.ออโต้เทรด (สำนักงานใหญ่),2,5220.00
88,ห้างหุ้นส่วนจำกัด ฮ.อาหลั่ยยนต์ (สำนักงานใหญ่),3,10036.60
89,ห้างหุ้นส่วนจำกัด เอส.เค.ที. แมชชีนทูลส์ (สนญ.),39,83421.67


In [41]:
def add_total_row(df, label_col="ACCTNAME", amount_col="total_amount"):
    total_row = pd.DataFrame([{
        label_col: "TOTAL",
        "bills": df["bills"].sum(),
        amount_col: df[amount_col].sum()
    }])

    return pd.concat([df, total_row], ignore_index=True)

In [42]:
def rename_and_keep(df, rename_map):
    cols = [c for c in rename_map.keys() if c in df.columns]
    return df[cols].rename(columns=rename_map)

In [43]:
df_ap_summary = add_total_row(df_ap_summary)
df_ar_summary = add_total_row(df_ar_summary)

In [44]:
rename_map = {
    "ACCTNAME": "ชื่อลูกหนี้",
    "BILLNO": "เลขที่บิล",
    "BILLDATE": "วันที่",
    "DUEAMT": "จำนวน",
    "bills": "จำนวนบิล",
    "total_amount": "ยอดรวม",
}

df_ar_sale     = rename_and_keep(df_ar_sale, rename_map)
df_ar_summary  = rename_and_keep(df_ar_summary, rename_map)

rename_map["ACCTNAME"] = "ชื่อเจ้าหนี้"

df_ap_purchase      = rename_and_keep(df_ap_purchase , rename_map)
df_ap_summary    = rename_and_keep(df_ap_summary  , rename_map)

In [45]:
folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "04_outputs",
    "03_AR_AP_Report",
    f"ar_ap_report_{YEAR}_{MONTH:02}.xlsx"
)

# ⭐ create directory if missing
os.makedirs(os.path.dirname(folder), exist_ok=True)

In [46]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

def export_ar_ap_report(
    df_ap_purchase,
    df_ap_summary,
    df_ar_sale,
    df_ar_summary,
    output_file,
):
    sheets = {
        "AP": df_ap_purchase.copy(),
        "AP Summary": df_ap_summary.copy(),
        "AR": df_ar_sale.copy(),
        "AR Summary": df_ar_summary.copy(),
    }

    wb = Workbook()
    wb.remove(wb.active)

    # styles
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(name="Tahoma", size=12, bold=True, color="FFFFFF")
    body_font = Font(name="Tahoma", size=11, color="000000")
    total_font = Font(name="Tahoma", size=11, bold=True, color="000000")

    thin_gray = Side(style="thin", color="D9D9D9")
    header_border = Border(bottom=Side(style="medium", color="FFFFFF"))
    body_border = Border(bottom=thin_gray)

    fill_group_1 = PatternFill("solid", fgColor="F7FBFF")
    fill_group_2 = PatternFill("solid", fgColor="E8F1FB")

    header_align = Alignment(horizontal="center", vertical="center")
    text_align = Alignment(horizontal="left", vertical="center")
    num_align = Alignment(horizontal="right", vertical="center")
    date_align = Alignment(horizontal="center", vertical="center")

    amount_headers = {"จำนวน", "ยอดรวม", "DUEAMT", "total_amount", "VAT"}
    date_headers = {"วันที่", "BILLDATE", "VOUCDATE2"}

    for sheet_name, df in sheets.items():
        ws = wb.create_sheet(title=sheet_name)

        # write dataframe
        for r_idx, row in enumerate(
            [list(df.columns)] + df.values.tolist(), start=1
        ):
            for c_idx, value in enumerate(row, start=1):
                cell = ws.cell(row=r_idx, column=c_idx, value=value)

                if r_idx == 1:
                    cell.fill = header_fill
                    cell.font = header_font
                    cell.alignment = header_align
                    cell.border = header_border
                else:
                    cell.font = body_font
                    cell.border = body_border

        # alternate row color by ACCTNAME / name group
        name_cols = ["ชื่อลูกหนี้", "ชื่อเจ้าหนี้", "ACCTNAME"]
        name_col = next((c for c in name_cols if c in df.columns), None)

        if name_col is not None and len(df) > 0:
            current_fill = fill_group_1
            prev_name = None

            name_col_idx = list(df.columns).index(name_col) + 1

            for row_idx in range(2, len(df) + 2):
                current_name = ws.cell(row=row_idx, column=name_col_idx).value

                if prev_name is None:
                    prev_name = current_name
                elif current_name != prev_name:
                    current_fill = fill_group_2 if current_fill == fill_group_1 else fill_group_1
                    prev_name = current_name

                for col_idx in range(1, len(df.columns) + 1):
                    ws.cell(row=row_idx, column=col_idx).fill = current_fill


        # format body cells by column type
        for col_idx, col_name in enumerate(df.columns, start=1):
            col_letter = get_column_letter(col_idx)

            # detect total row if present
            total_row_idx = None
            if len(df) > 0 and col_name in ["ชื่อลูกหนี้", "ชื่อเจ้าหนี้", "ACCTNAME"]:
                for i, v in enumerate(df[col_name].astype(str), start=2):
                    if v.strip().upper() == "TOTAL":
                        total_row_idx = i
                        break

            for row_idx in range(2, len(df) + 2):
                cell = ws.cell(row=row_idx, column=col_idx)

                if col_name in date_headers:
                    if pd.notna(cell.value):
                        try:
                            dt = pd.to_datetime(cell.value)
                            cell.value = dt.to_pydatetime()
                            # Thai locale format; no time
                            cell.number_format = '[$-th-TH]d mm yyyy'
                        except Exception:
                            pass
                    cell.alignment = date_align

                elif col_name in amount_headers:
                    cell.number_format = '#,##0.00;[Red](#,##0.00);-'
                    cell.alignment = num_align

                elif pd.api.types.is_numeric_dtype(df[col_name]):
                    cell.number_format = '#,##0.00;[Red](#,##0.00);-'
                    cell.alignment = num_align

                else:
                    cell.alignment = text_align

            # bold total row if present
            if total_row_idx is not None:
                ws.cell(total_row_idx, col_idx).font = total_font

        # row heights
        ws.row_dimensions[1].height = 24
        for r in range(2, len(df) + 2):
            ws.row_dimensions[r].height = 20

        # improved auto-fit column widths
        for col_idx, col_name in enumerate(df.columns, start=1):
            col_letter = get_column_letter(col_idx)

            max_len = len(str(col_name))

            for row_idx in range(2, len(df) + 2):
                val = ws.cell(row=row_idx, column=col_idx).value
                if val is None:
                    continue

                text = str(val)

                # Thai characters are visually wider
                visual_len = len(text) + sum(1 for ch in text if ord(ch) > 127)
                max_len = max(max_len, visual_len)

            # special handling for name column
            if col_name in ["ชื่อลูกหนี้", "ชื่อเจ้าหนี้", "ACCTNAME"]:
                ws.column_dimensions[col_letter].width = min(max_len + 6, 80)

            elif col_name in ["เลขที่บิล", "BILLNO"]:
                ws.column_dimensions[col_letter].width = min(max_len + 4, 24)

            elif col_name in ["วันที่", "BILLDATE", "VOUCDATE2"]:
                ws.column_dimensions[col_letter].width = 16

            else:
                ws.column_dimensions[col_letter].width = min(max_len + 3, 22)

        # freeze header
        ws.freeze_panes = "A2"

        # add Excel table for filters + nicer readability
        if len(df.columns) > 0 and len(df) > 0:
            end_col = get_column_letter(len(df.columns))
            end_row = len(df) + 1
            table_ref = f"A1:{end_col}{end_row}"

            table = Table(displayName=f"Tbl_{sheet_name.replace(' ', '_')}", ref=table_ref)
            style = TableStyleInfo(
                name="TableStyleMedium2",
                showFirstColumn=False,
                showLastColumn=False,
                showRowStripes=True,
                showColumnStripes=False,
            )
            table.tableStyleInfo = style
            ws.add_table(table)

    wb.save(output_file)

In [47]:
output_file = folder

export_ar_ap_report(
    df_ap_purchase=df_ap_purchase,
    df_ap_summary=df_ap_summary,
    df_ar_sale=df_ar_sale,
    df_ar_summary=df_ar_summary,
    output_file=output_file,
)